In [ ]:
from datasets import load_dataset, Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from PIL import Image
import os
import evaluate
import numpy as np

/home/exouser/colrc-ocr-model/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "microsoft/trocr-base-printed"
processor = TrOCRProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
special_tokens = ["č", "ˡɫ", "ʷ","ᵃ\u0308", "u̥","ᵘ", "ɔ", "ä", "ĺ", "ý", "ɛ", "x̥","ʙ", "ẃ", "q́","ḿ", "ˠ", "‿", "t́", "ʀ"  ]  # add as needed
processor.tokenizer.add_tokens(special_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


TrOCRScaledWordEmbedding(50284, 1024, padding_idx=1)

In [4]:
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id or processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

In [ ]:

def load_samples(path):
    data = {"image": [], "text": []}
    for fname in os.listdir(path):
        if fname.endswith(".png"):
            with open(os.path.join(path, fname.replace(".png", ".txt"))) as f:
                data["image"].append(Image.open(os.path.join(path, fname)).convert("RGB"))
                data["text"].append(f.read().strip())
    return data

train_data = load_samples("../cda/train")
val_data = load_samples("../cda/val")


train_ds = Dataset.from_dict(train_data)
val_ds = Dataset.from_dict(val_data)

In [6]:
def preprocess(batch):
    pixel_values = processor(images=batch["image"], return_tensors="pt").pixel_values
    labels = processor.tokenizer(batch["text"], padding="max_length", truncation=True, return_tensors="pt").input_ids
    return {"pixel_values": pixel_values.squeeze(), "labels": labels.squeeze()}

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)
train_ds.set_format(type="torch", columns=["pixel_values", "labels"])
val_ds.set_format(type="torch", columns=["pixel_values", "labels"])

Map: 100%|██████████| 600/600 [03:28<00:00,  2.88 examples/s]


In [19]:

training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-cda",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    predict_with_generate=True,
    fp16=True
)




In [ ]:
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

In [ ]:

#Used ChatGPT to help fix this function because I was getting a float type error
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    if pred_ids.dtype != np.int64 and pred_ids.dtype != np.int32:
        pred_ids = np.argmax(pred_ids, axis=-1)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer, "wer": wer}


In [23]:
from transformers import default_data_collator
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator
)


In [24]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss,Cer,Wer
1,0.208100,0.203990,0.752069,1.014550
2,0.154600,0.140935,0.651173,0.994684
3,0.121600,0.120322,0.618211,0.953274
4,0.077100,0.064984,0.600129,0.941242
5,0.043800,0.040507,0.623966,0.951035
6,0.025100,0.022212,0.572752,0.956351
7,0.015100,0.012585,0.587428,0.963067
8,0.007200,0.008533,0.582933,0.950196
9,0.003500,0.005743,0.572513,0.949356
10,0.002200,0.004314,0.568529,0.941522


TrainOutput(global_step=7500, training_loss=0.09079867376287779, metrics={'train_runtime': 2764.5092, 'train_samples_per_second': 10.852, 'train_steps_per_second': 2.713, 'total_flos': 2.244855572987904e+19, 'train_loss': 0.09079867376287779, 'epoch': 10.0})